# Primary Econometric Analysis

## 1. Declaration

The empirical analysis in this study is descriptive. The results are interpreted as conditional correlations and predictive associations between international student enrolments and rental prices, rather than causal effects. The analysis uses a regression framework with controls to account for observable factors, but does not establish a credible identification strategy. Therefore, the results should not be interpreted as evidence of a causal effect.

## 2. Econometric Specifiation

$$
\Delta \ln(RentalPrice_{it}) = \beta_0 + \beta_1 \Delta \ln(InternationalStudents_{it}) + \beta_2 \Delta \ln(Population_{it}) + \beta_3 \Delta \ln(HousingSupply_{it}) + \alpha_i + \lambda COVID_t + \varepsilon_{it}
$$

**Functional Form**

1. Log-differences (Δln) 
- Variables expressed as quarterly percentage changes
- Coefficients interpreted as elasticities in changes
- First-differencing helps remove common time trends

**Regressors**

1. Main Variable
- Δln(InternationalStudents₍ᵢₜ₎)

2. Control Variables
- Δln(Population₍ᵢₜ₎)  
- Δln(Supply₍ᵢₜ₎)  

3. Fixed Effects
- αᵢ (state fixed effects)

4. Dummy Variable 
- COVIDₜ  

**Sample**

1. Panel data
- i: states (NSW, VIC)
- t: time (quarters, 2019–2024)

**Error structure**

- Omitted variables
- Measurement error 
- Idiosyncratic shocks

## 3. Regression Table

In [10]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort properly
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# First differences
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# COVID dummy
df["COVID"] = ((df["Date"] >= "2020-Q1") & (df["Date"] <= "2022-Q1")).astype(int)

# Drop NA
df = df.dropna()

# -------------------------
# MODEL 1: Students only
# -------------------------
model1 = smf.ols(
    "Δ_ln_rent ~ Δ_ln_students",
    data=df
).fit()

# -------------------------
# MODEL 2: FULL MODEL
# -------------------------
model2 = smf.ols(
    """
    Δ_ln_rent ~ Δ_ln_students 
              + Δ_ln_population 
              + Δ_ln_supply 
              + COVID 
              + C(State)
    """,
    data=df
).fit()

# Print results
print("=========================== MODEL 1: Students Only ===========================")
print(model1.summary())

print("============================ MODEL 2: Final Model ============================")
print(model2.summary())

=========================== MODEL 1: Students Only ===========================
                            OLS Regression Results                            
Dep. Variable:              Δ_ln_rent   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.8339
Date:                Tue, 05 May 2026   Prob (F-statistic):              0.366
Time:                        12:55:16   Log-Likelihood:                 129.42
No. Observations:                  46   AIC:                            -254.8
Df Residuals:                      44   BIC:                            -251.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------

| Variable        | Model 1: Students Only | Model 2: Full Model |
| --------------- | ---------------------: | ------------------: |
| Constant        |                 0.0120 |              0.0295 |
|                 |                (0.002) |             (0.010) |
| Δ ln Students   |                 0.0012 |              0.0003 |
|                 |                (0.001) |             (0.001) |
| Δ ln Population |                      — |              1.3973 |
|                 |                      — |             (0.948) |
| Δ ln Supply     |                      — |             -4.1544 |
|                 |                      — |             (2.349) |
| COVID           |                      — |             -0.0115 |
|                 |                      — |             (0.005) |
| Observations    |                     46 |                  46 |
| R²              |                  0.019 |               0.565 |

## 4. Interpretation

The coefficient of 0.0003 on changes of log international studet enrolments suggests a very small positive relationship between changes in international student enrolments and changes in rental prices. An increase of 1% in international student enrolments is associated with a 0.0003% increase in rental prices, holding changes in population, changes in housing supply, the COVID-19 period effect and time-invariant differences between states constant. This effect is extremely small and statistically insignificant (p = 0.812) which shows no reliable or zero relationship between changes in international student numbers and rental price growth. In terms of units, both the dependent and independent variables are expressed in log differences, so the coefficient can be interpreted as an elasticity of changes, a percentage change in rental prices in response to a percentage change in student enrolments.

## 5. Threats

1. Omitted Variable Bias
- Model does not include all factors affecting rental prices.
- Variables such as interest rates, inflation, housing policy, broader migration are missing.
- These factors may be correlated with both rent and student enrolments
- May bias estimated relationships.

2. Aggregation Bias
- State-level data masks within-state variation.
- Rental markets differ across locations in the cities, suburbs, regional.
- International students are concentrated in urban areas.
- May weaken or obscure the true relationship.

3. Multicollinearity
- Student enrolments are closely related to population growth.
- Creates correlation between regressors.
- Makes it difficult to isolate independent effects.
- Coefficients may become unstable or statistically insignificant.

4. Measurement Error
- Differencing may introduce noise.
- Timing mismatch between student arrivals and rental adjustments.
- May weaken the observed relationship.
